# PCD_Assignment03
## Otsu Segmentation and Morphological Operations

**Name:** Aditya Rahman  
**Student ID:** 25/554909/PA/23248

## 1. Image Input

Pillow is used only to read the JPEG file and extract its RGB pixel values. Grayscale conversion, Otsu thresholding, morphological operations, and comparison statistics are implemented manually using Python loops and arithmetic. No library functions are used for segmentation or morphology.

Image previews are constructed manually as SVG markup and rendered by the notebook interface, without a plotting library.

In [1]:
from PIL import Image  # Used only to read the JPEG file.

IMAGE_PATH = '/content/cartman.jpeg'

with Image.open(IMAGE_PATH) as decoded:
    decoded.load()
    if decoded.mode != 'RGB':
        raise ValueError('Expected an RGB JPEG image.')
    width, height = decoded.size
    pixels = list(decoded.getdata())

rgb = [[pixels[y * width + x] for x in range(width)] for y in range(height)]
print('Image dimensions:', width, 'x', height)

Image dimensions: 328 x 302


## 2. Grayscale conversion

Each RGB pixel is converted to one intensity value using

$$I(x,y)=0.299R(x,y)+0.587G(x,y)+0.114B(x,y).$$

The weighted sum is rounded to an integer between 0 and 255. Grayscale allows Otsu thresholding to work with a single intensity histogram. The original and grayscale previews below are generated from the pixel values; only the previews are sampled at two-pixel intervals to keep the notebook compact. Every processing operation uses the full-resolution image.

In [2]:
class ImagePanels:
    def __init__(self, items, step=2):
        self.items = items
        self.step = step

    def _repr_html_(self):
        # Build SVG rectangles manually. No image or plotting package is used.
        panels = ['<div style="display:flex;flex-wrap:wrap;gap:16px">']
        for title, image, mode in self.items:
            h, w = len(image), len(image[0])
            step = self.step
            sw, sh = (w + step - 1) // step, (h + step - 1) // step
            title = title.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')
            panels.append('<div><p style="font-weight:bold">' + title + '</p>')
            panels.append('<svg xmlns="http://www.w3.org/2000/svg" width="328" height="302" viewBox="0 0 '+str(sw)+' '+str(sh)+'" style="background:black;image-rendering:pixelated" shape-rendering="crispEdges">')
            for sy, y in enumerate(range(0, h, step)):
                colors = []
                for x in range(0, w, step):
                    value = image[y][x]
                    if mode == 'rgb':
                        r, g, b = value
                    else:
                        r = g = b = value * 255 if mode == 'binary' else value
                    colors.append('#'+format(r, '02x')+format(g, '02x')+format(b, '02x'))
                start = 0
                while start < sw:
                    end = start + 1
                    while end < sw and colors[end] == colors[start]:
                        end += 1
                    panels.append('<rect x="'+str(start)+'" y="'+str(sy)+'" width="'+str(end-start)+'" height="1" fill="'+colors[start]+'"/>')
                    start = end
            panels.append('</svg></div>')
        panels.append('</div>')
        return ''.join(panels)

gray = [[int(round(0.299*r + 0.587*g + 0.114*b)) for r, g, b in row] for row in rgb]
ImagePanels([('Original image', rgb, 'rgb'), ('Grayscale', gray, 'gray')])

## 3. Otsu segmentation

Otsu evaluates thresholds that divide intensities into $[0,t]$ and $[t+1,255]$. It selects the threshold with the highest between-class variance:

$$\sigma_B^2(t)=w_0(t)w_1(t)(\mu_0(t)-\mu_1(t))^2.$$

The histogram and cumulative counts are calculated manually. Empty classes are skipped, and the first maximizing threshold is retained when scores tie. Histogram construction costs $O(HW)$, and the threshold search costs $O(256)$.

Since the background is white, pixels with intensity **less than or equal to** the threshold are assigned foreground (1). Foreground is displayed in white. This separates darker image regions; it does not identify the whole character semantically. Bright facial regions can be excluded along with the background.

In [3]:
def otsu_threshold(image):
    if not image or not image[0]:
        raise ValueError('Image must be nonempty.')
    width = len(image[0])
    histogram = [0] * 256
    for row in image:
        if len(row) != width:
            raise ValueError('Image must be rectangular.')
        for value in row:
            if not isinstance(value, int) or not 0 <= value <= 255:
                raise ValueError('Expected integer intensities in [0, 255].')
            histogram[value] += 1
    total = len(image) * width
    total_sum = sum(level * count for level, count in enumerate(histogram))
    background_count = 0
    background_sum = 0
    best_score = -1.0
    threshold = next(i for i, count in enumerate(histogram) if count)
    for t in range(256):
        background_count += histogram[t]
        background_sum += t * histogram[t]
        foreground_count = total - background_count
        if background_count == 0 or foreground_count == 0:
            continue
        mean0 = background_sum / background_count
        mean1 = (total_sum - background_sum) / foreground_count
        score = (background_count / total) * (foreground_count / total) * (mean0 - mean1)**2
        if score > best_score:
            best_score = score
            threshold = t
    return threshold, histogram

def threshold_image(image, threshold):
    return [[int(value > threshold) for value in row] for row in image]



threshold, histogram = otsu_threshold(gray)
raw = [[int(value <= threshold) for value in row] for row in gray]
print('Otsu threshold:', threshold)
ImagePanels([('Grayscale input', gray, 'gray'), ('Otsu segmentation', raw, 'binary')])

Otsu threshold: 169


## 4. Morphological operations

A centered, all-ones $3\times3$ square is used as the structuring element. Pixels outside the image are treated as background (0). This padding can shrink foreground regions touching the border.

| Operation | Neighborhood rule | Typical effect |
|---|---|---|
| Erosion | All covered pixels must be 1 | Shrinks regions; removes thin features |
| Dilation | At least one covered pixel must be 1 | Expands regions; bridges small gaps |
| Opening | Erosion followed by dilation | Removes small foreground structures |
| Closing | Dilation followed by erosion | Fills small background gaps |

Each primitive uses nested loops and takes $O(HWk^2)$ time for a $k\times k$ element, with $O(HW)$ output memory. All alternatives below start from the same Otsu result.

In [4]:
def binary_morphology(mask, size, operation):
    if not isinstance(size, int) or size < 1 or size % 2 == 0:
        raise ValueError('Size must be a positive odd integer.')
    if operation not in ('erosion', 'dilation'):
        raise ValueError('Unknown operation.')
    if not mask or not mask[0]:
        raise ValueError('Mask must be nonempty.')
    height, width = len(mask), len(mask[0])
    if any(len(row) != width or any(v not in (0, 1) for v in row) for row in mask):
        raise ValueError('Expected a rectangular binary mask.')
    radius = size // 2
    result = [[0] * width for _ in range(height)]
    for y in range(height):
        for x in range(width):
            value = 1 if operation == 'erosion' else 0
            decided = False
            for dy in range(-radius, radius + 1):
                for dx in range(-radius, radius + 1):
                    yy, xx = y + dy, x + dx
                    pixel = mask[yy][xx] if 0 <= yy < height and 0 <= xx < width else 0
                    if operation == 'erosion' and pixel == 0:
                        value, decided = 0, True
                        break
                    if operation == 'dilation' and pixel == 1:
                        value, decided = 1, True
                        break
                if decided:
                    break
            result[y][x] = value
    return result

def erosion(mask, size=3):
    return binary_morphology(mask, size, 'erosion')

def dilation(mask, size=3):
    return binary_morphology(mask, size, 'dilation')

def opening(mask, size=3):
    return dilation(erosion(mask, size), size)

def closing(mask, size=3):
    return erosion(dilation(mask, size), size)



KERNEL_SIZE = 3
eroded = erosion(raw, KERNEL_SIZE)
dilated = dilation(raw, KERNEL_SIZE)
opened = opening(raw, KERNEL_SIZE)
closed = closing(raw, KERNEL_SIZE)
processed = closing(opened, KERNEL_SIZE)
ImagePanels([
    ('Otsu only', raw, 'binary'), ('Erosion', eroded, 'binary'),
    ('Dilation', dilated, 'binary'), ('Opening', opened, 'binary'),
    ('Closing', closed, 'binary'), ('Opening then closing', processed, 'binary')
])

## 5. Implementation verification

Small masks with known outcomes check the neighborhood operations, opening and closing, the border convention, and Otsu threshold selection. No reference library functions are used.

In [5]:
single = [[0] * 7 for _ in range(7)]
single[3][3] = 1
expected_square = [[int(2 <= y <= 4 and 2 <= x <= 4) for x in range(7)] for y in range(7)]
assert dilation(single) == expected_square
assert sum(map(sum, erosion(single))) == 0
assert erosion(expected_square) == single
assert opening(single) == [[0] * 7 for _ in range(7)]
block = [[int(1 <= y <= 5 and 1 <= x <= 5) for x in range(7)] for y in range(7)]
pinhole = [row[:] for row in block]
pinhole[3][3] = 0
assert closing(pinhole) == block
assert erosion(raw, 1) == raw == dilation(raw, 1)
assert opening(opened) == opened
assert closing(closed) == closed
assert otsu_threshold([[10, 10, 200, 200]])[0] == 10
assert threshold_image([[10, 10, 200, 200]], 10) == [[0, 0, 1, 1]]
assert threshold_image([[50, 50]], otsu_threshold([[50, 50]])[0]) == [[0, 0]]
assert erosion([[1]]) == [[0]]  # zero extension at image borders
assert dilation([[1]]) == [[1]]
print('All implementation checks passed.')

All implementation checks passed.


## 6. Comparison with and without morphology

The baseline and processed results use the same image and threshold. Foreground area, 4-connected component count, and added/removed pixels describe the effects of morphology. Component counting uses a manually implemented flood fill with four neighbors: up, down, left, and right.

No annotated reference mask is available, so these are descriptive measurements rather than accuracy scores. A smaller component count may indicate removal of details or merging of regions, and does not automatically imply better segmentation.

In [6]:
def component_count(mask):
    height, width = len(mask), len(mask[0])
    visited = [[False] * width for _ in range(height)]
    count = 0
    for y in range(height):
        for x in range(width):
            if not mask[y][x] or visited[y][x]:
                continue
            count += 1
            stack = [(y, x)]
            visited[y][x] = True
            while stack:
                yy, xx = stack.pop()
                for dy, dx in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    ny, nx = yy + dy, xx + dx
                    if 0 <= ny < height and 0 <= nx < width and mask[ny][nx] and not visited[ny][nx]:
                        visited[ny][nx] = True
                        stack.append((ny, nx))
    return count

def summarize(mask, baseline):
    foreground = sum(sum(row) for row in mask)
    added = removed = 0
    for row, original in zip(mask, baseline):
        for p, q in zip(row, original):
            added += int(p == 1 and q == 0)
            removed += int(p == 0 and q == 1)
    return {'area':foreground, 'percent':100 * foreground / (len(mask) * len(mask[0])),
            'components':component_count(mask), 'added':added, 'removed':removed}

assert component_count([[1, 0], [0, 1]]) == 2
assert component_count([[1, 1], [0, 1]]) == 1
assert component_count([[0]]) == 0
variants = {'Otsu only':raw, 'Erosion':eroded, 'Dilation':dilated,
            'Opening':opened, 'Closing':closed, 'Opening + closing':processed}
statistics = {name:summarize(mask, raw) for name, mask in variants.items()}
print(f"{'Method':<21} {'FG pixels':>10} {'FG %':>8} {'Components':>11} {'Added':>8} {'Removed':>8}")
for name, s in statistics.items():
    print(f"{name:<21} {s['area']:10d} {s['percent']:8.2f} {s['components']:11d} {s['added']:8d} {s['removed']:8d}")


before, after = statistics['Otsu only'], statistics['Opening + closing']
print('Components:', before['components'], '->', after['components'])
print('Foreground area:', before['area'], '->', after['area'])
print('Pixels added:', after['added'], '| Pixels removed:', after['removed'])

changes = [[((0, 204, 51) if p and not q else (255, 38, 26) if q and not p
             else (255, 255, 255) if p else (0, 0, 0)) for p, q in zip(row, original)]
           for row, original in zip(processed, raw)]
ImagePanels([('Without morphology', raw, 'binary'),
             ('Opening then closing', processed, 'binary'),
             ('Green: added / Red: removed', changes, 'rgb')])

Method                 FG pixels     FG %  Components    Added  Removed
Otsu only                  39365    39.74          10        0        0
Erosion                    36755    37.11           7        0     2610
Dilation                   42252    42.65           7     2887        0
Opening                    38897    39.27           5        0      468
Closing                    39388    39.76           9       23        0
Opening + closing          38914    39.28           5       15      466
Components: 10 -> 5
Foreground area: 39365 -> 38914
Pixels added: 15 | Pixels removed: 466


## 7. Kernel size comparison

Increasing the square element from 3×3 to 5×5 applies stronger spatial processing. Larger elements can remove more narrow features while also discarding useful details. The threshold and original mask are held fixed in this comparison.

In [7]:
large_kernel = closing(opening(raw, 5), 5)
for size, mask in [(3, processed), (5, large_kernel)]:
    s = summarize(mask, raw)
    print('Kernel:', size, 'x', size, '| Area:', s['area'], '| Components:', s['components'],
          '| Added:', s['added'], '| Removed:', s['removed'])
ImagePanels([('Otsu only', raw, 'binary'), ('3 x 3 element', processed, 'binary'),
             ('5 x 5 element', large_kernel, 'binary')])

Kernel: 3 x 3 | Area: 38914 | Components: 5 | Added: 15 | Removed: 466
Kernel: 5 x 5 | Area: 38691 | Components: 5 | Added: 3 | Removed: 677


## 8. Discussion and conclusion

Otsu thresholding segments darker regions of the Cartman image. It excludes some bright character regions, including much of the face, because a single brightness threshold does not represent semantic object boundaries.

Without morphology, all narrow details and small components in the initial thresholded mask remain. Erosion shrinks the mask, while dilation expands it. Opening removes small foreground features, and closing bridges small background gaps. Facial or clothing details removed by these operations may be valid structures rather than noise.

For the 3×3 opening-then-closing pipeline, the foreground area changes from **39,365 to 38,914 pixels**, and the number of 4-connected components changes from **10 to 5**. The result adds **15 pixels** and removes **466 pixels** relative to the baseline. Thus, morphology changes connectivity and removes thin features, but these statistics do not prove improved accuracy.

The larger 5×5 element removes more detail. Morphology cannot recover large bright regions excluded by the original segmentation. A reference mask would be required to measure segmentation quality using IoU or Dice.

The image depicts Cartman from South Park and was supplied as the experiment input.